# NB20 — Coreness-Matched Permutation Test

**Purpose:** Validate that the primary PGLS signal cannot be explained by KO coreness
(the tendency for widely-distributed, "core" genes to be in both our gene list and
large-genome, generalist taxa).

**Definition — annotation depth (for clarity):**
- *Annotation depth* (used in Methods): mean number of genomes per genus that carry
  at least one KO from the target gene set. This is a per-genus summary, not a
  per-KO property.
- *KO coreness*: fraction of all genomes in kbase.ke_pangenome that carry a given KO.
  This is a per-KO property that reflects pan-genome prevalence.

**Why coreness matters:** Core genes (present in most genomes) may covary with
genome size and niche breadth independently of metal ecology. If our primary KO set
is enriched for core KOs, a non-specific coreness effect could drive the signal.

**Design:** 1,000 permuted KO sets, each matched to the primary set (n=140 KOs) on
both KO count AND coreness distribution (decile matching). Run identical PGLS pipeline
on each; report empirical p = fraction with β ≤ observed.
Secondary: 100 sets with genome-size covariate; compute % β-attenuation vs. primary.

**Requires:** JupyterHub (Spark). Label: Exploratory. Run once.

**Outputs:**
- `data/coreness_permutation_results.csv`
- `data/attenuation_profile_comparison.csv`
- `figures/coreness_permutation_histogram.png`


In [1]:
import sys, random
from pathlib import Path
import pandas as pd
import numpy as np
from statsmodels.stats.multitest import multipletests

PROJECT = Path('/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology')
DATA    = PROJECT / 'data'
FIGS    = PROJECT / 'figures'
TREE_BAC = DATA / 'gtdb_bac_genus_pruned.tree'

sys.path.insert(0, str(PROJECT / 'scripts'))
from pgls_utils import run_pgls

_SPARK_AVAILABLE = False
_spark = None
try:
    from berdl_utils import get_spark_session
    _spark = get_spark_session()
    _SPARK_AVAILABLE = True
    print('Spark OK')
except BaseException as _e:
    print(f'Spark unavailable: {_e}')

bac_base  = pd.read_csv(DATA / '01_pgls_input_bacteria.csv')
trait_df  = bac_base[['genus_lower', 'mean_levins_B_std']].copy()
print(f'Base data: {len(bac_base)} genera')

gene_df = pd.read_csv(DATA / 'curated_mrg_ko_ids_v2.csv')
primary_kos = list(gene_df[gene_df['evidence_tier'].isin(['Tier 1','Tier 2'])]['KO'])
print(f'Primary KOs: {len(primary_kos)}')
OBS_BETA = -0.021   # P1 result
OBS_SE   = 0.0037
N_TARGET = len(primary_kos)


[berdl_utils] JupyterHub SparkSession acquired: 4.0.1
Spark OK
Base data: 1574 genera
Primary KOs: 140


## Block 2 — Compute KO coreness from kbase.ke_pangenome

In [9]:
if not _SPARK_AVAILABLE:
    raise RuntimeError('Spark required — run in JupyterHub')

# ------------------------------------------------------------
# 1. List all tables in the schema
# ------------------------------------------------------------
schema = "kbase.ke_pangenome"
print(f"Tables in {schema}:")
_spark.sql(f"SHOW TABLES IN {schema}").show(50, truncate=False)

# ------------------------------------------------------------
# 2. Find a table with a KO column
# ------------------------------------------------------------
# Get all table names in the schema
tables = [row.tableName for row in _spark.sql(f"SHOW TABLES IN {schema}").collect()]
print(f"\nFound {len(tables)} tables.")

ko_table = None
ko_col = None

for t in tables:
    full_name = f"{schema}.{t}"
    try:
        cols = [row['col_name'] for row in _spark.sql(f"DESCRIBE {full_name}").collect()]
        # Look for a column that likely holds KO identifiers
        for col in cols:
            if 'ko' in col.lower() or 'kegg' in col.lower() or 'k_number' in col.lower():
                ko_table = t
                ko_col = col
                break
        if ko_table:
            break
    except Exception as e:
        print(f"Could not describe {full_name}: {e}")

if ko_table is None:
    raise RuntimeError("No table with a KO column found in the schema.")

print(f"\n✅ Found KO column '{ko_col}' in table '{ko_table}'.")

# ------------------------------------------------------------
# 3. Join gene table with the KO annotation table
# ------------------------------------------------------------
# The gene table has (gene_id, genome_id). The annotation table likely has (gene_id, ko_col).
# We need to handle possible multiple annotations per gene (e.g., many-to-many).
# Assume we join on gene_id.

gene_table = f"{schema}.gene"
anno_table = f"{schema}.{ko_table}"

# Check if annotation table has a gene_id column; if not, we may need a different join key.
# Let's describe it to see columns.
print(f"\nColumns in {anno_table}:")
cols_anno = [row['col_name'] for row in _spark.sql(f"DESCRIBE {anno_table}").collect()]
print(cols_anno)

# If 'gene_id' is not present, we might need to join via another table (e.g., gene_cluster).
# For now, assume gene_id is the foreign key.
join_col = 'gene_id'
if join_col not in cols_anno:
    # Try alternative: maybe 'gene_id' is called something else, or we need a bridge table.
    # We'll search for a column that likely refers to gene.
    potential_join = [c for c in cols_anno if 'gene' in c.lower()]
    if potential_join:
        join_col = potential_join[0]
        print(f"Using join column '{join_col}' instead of 'gene_id'.")
    else:
        print("No gene link found in annotation table. Trying to join via other tables...")
        # Option: if annotation table has protein_id, and there is a gene_protein table, etc.
        # For now, raise a clear message.
        raise RuntimeError(f"Could not find a gene identifier in {anno_table}. Need to inspect schema.")

# ------------------------------------------------------------
# 4. Compute per‑KO prevalence
# ------------------------------------------------------------
# Build query: get distinct (genome_id, ko) pairs, then count genomes per ko
prevalence_sql = f"""
    SELECT 
        a.{ko_col} AS ko,
        COUNT(DISTINCT g.genome_id) AS n_genomes
    FROM {gene_table} g
    JOIN {anno_table} a ON g.gene_id = a.{join_col}
    WHERE a.{ko_col} IS NOT NULL AND a.{ko_col} != ''
    GROUP BY a.{ko_col}
"""
prevalence_df = _spark.sql(prevalence_sql).toPandas()
print(f"\n✅ Per-KO prevalence computed: {len(prevalence_df)} distinct KOs found.")
print(prevalence_df.head(10))

# (Optional) save to CSV
# prevalence_df.to_csv(DATA / 'ko_prevalence.csv', index=False)

Tables in kbase.ke_pangenome:
+------------+--------------------------------+-----------+
|namespace   |tableName                       |isTemporary|
+------------+--------------------------------+-----------+
|ke_pangenome|alphaearth_embeddings_all_years |false      |
|ke_pangenome|bakta_amr                       |false      |
|ke_pangenome|bakta_annotations               |false      |
|ke_pangenome|bakta_db_xrefs                  |false      |
|ke_pangenome|bakta_pfam_domains              |false      |
|ke_pangenome|eggnog_mapper_annotations       |false      |
|ke_pangenome|gapmind_pathways                |false      |
|ke_pangenome|gene                            |false      |
|ke_pangenome|gene_cluster                    |false      |
|ke_pangenome|gene_genecluster_junction       |false      |
|ke_pangenome|genome                          |false      |
|ke_pangenome|genome_ani                      |false      |
|ke_pangenome|gtdb_metadata                   |false      |
|ke_pangen


✅ Found KO column 'kegg_orthology_id' in table 'bakta_annotations'.

Columns in kbase.ke_pangenome.bakta_annotations:
['gene_cluster_id', 'length', 'gene', 'product', 'hypothetical', 'ec', 'go', 'cog_id', 'cog_category', 'kegg_orthology_id', 'refseq', 'uniparc', 'uniref100', 'uniref90', 'uniref50', 'molecular_weight', 'isoelectric_point']
Using join column 'gene_cluster_id' instead of 'gene_id'.



✅ Per-KO prevalence computed: 6680 distinct KOs found.
       ko  n_genomes
0  K09991       1694
1  K06042       8333
2  K06879       3669
3  K00198       2272
4  K07175       9312
5  K06145       3723
6  K08176        314
7  K01952      15632
8  K01685       4258
9  K07099       3440


In [10]:
# ------------------------------------------------------------
# 1. Find the annotation table that has a KO column
# ------------------------------------------------------------
schema = "kbase.ke_pangenome"
tables = [row.tableName for row in _spark.sql(f"SHOW TABLES IN {schema}").collect()]

ko_table = None
ko_col = None
gene_table = f"{schema}.gene"

for t in tables:
    full = f"{schema}.{t}"
    try:
        cols = [r['col_name'] for r in _spark.sql(f"DESCRIBE {full}").collect()]
        for c in cols:
            if 'ko' in c.lower() or 'kegg' in c.lower() or 'k_number' in c.lower():
                ko_table = t
                ko_col = c
                break
        if ko_table:
            break
    except:
        continue

if ko_table is None:
    raise RuntimeError("No table with a KO column found in the schema.")

print(f"✅ Using KO column '{ko_col}' from table '{ko_table}'.")

# Check if annotation table has gene_id; if not, find a linking column
anno_table = f"{schema}.{ko_table}"
anno_cols = [r['col_name'] for r in _spark.sql(f"DESCRIBE {anno_table}").collect()]
join_col = 'gene_id'
if join_col not in anno_cols:
    # Fallback: look for a column that likely references gene
    candidates = [c for c in anno_cols if 'gene' in c.lower() or 'protein' in c.lower()]
    if candidates:
        join_col = candidates[0]
        print(f"Using '{join_col}' as join column (instead of gene_id).")
    else:
        raise RuntimeError(f"Cannot find a gene identifier in {anno_table}. Inspect columns: {anno_cols}")

# ------------------------------------------------------------
# 2. Build query: join gene → annotation, extract KOs
# ------------------------------------------------------------
# We need to handle cases where ko_col may contain multiple KOs (e.g., comma‑separated).
# We'll use `split` and `explode` to get one KO per row.
# Also strip 'ko:' prefix if present, and keep only K[0-9]{5} pattern.

# First, get the distinct (genome_id, gene_id) from gene table.
# Then join with annotation on gene_id, and explode the KO string.

coreness_sql = f"""
WITH gene_ko AS (
    SELECT 
        g.genome_id,
        -- Split KO column on comma, trim, explode
        TRIM(ko_item) AS ko_raw
    FROM {gene_table} g
    JOIN {anno_table} a ON g.gene_id = a.{join_col}
    LATERAL VIEW EXPLODE(SPLIT(a.{ko_col}, ',')) t AS ko_item
    WHERE a.{ko_col} IS NOT NULL AND TRIM(a.{ko_col}) != ''
)
SELECT 
    -- Normalise to bare K-number (strip 'ko:' if present)
    REGEXP_EXTRACT(ko_raw, '(K[0-9]{{5}})', 1) AS ko,
    COUNT(DISTINCT genome_id) / {total_genomes}.0 AS coreness
FROM gene_ko
WHERE ko_raw RLIKE '(ko:)?K[0-9]{{5}}'
GROUP BY REGEXP_EXTRACT(ko_raw, '(K[0-9]{{5}})', 1)
HAVING ko IS NOT NULL AND ko != ''
"""

coreness_df = _spark.sql(coreness_sql).toPandas()

print(f'✅ Per-KO coreness computed: {len(coreness_df):,} unique KOs')
print(coreness_df['coreness'].describe())

# Save
coreness_df.to_csv(DATA / 'ko_coreness_pangenome.csv', index=False)
print('Saved: data/ko_coreness_pangenome.csv')


✅ Using KO column 'kegg_orthology_id' from table 'bakta_annotations'.
Using 'gene_cluster_id' as join column (instead of gene_id).


✅ Per-KO coreness computed: 6,680 unique KOs
count           6680
unique          3265
top       0.00001365
freq              63
Name: coreness, dtype: object
Saved: data/ko_coreness_pangenome.csv


## Block 3 — Coreness-matched permutation (1,000 sets)

In [12]:
# ------------------------------------------------------------
# Ensure coreness values are float
# ------------------------------------------------------------
coreness_df['coreness'] = coreness_df['coreness'].astype(float)

# Total number of primary KOs we want to match
N_TARGET = len(primary_kos)   # or len(primary_coreness) if some missing

# ------------------------------------------------------------
# 1. Get coreness for primary KOs
# ------------------------------------------------------------
ko_core = coreness_df.set_index('ko')['coreness'].to_dict()
primary_coreness = pd.Series({k: ko_core.get(k, np.nan) for k in primary_kos}).dropna()
print(f'Primary KOs with coreness data: {len(primary_coreness)} / {N_TARGET}')

# ------------------------------------------------------------
# 2. Decile bins from the full distribution of all KOs
# ------------------------------------------------------------
ALL_KO_CORE = coreness_df.set_index('ko')['coreness']
# Use np.percentile to avoid Decimal issues (already float)
decile_bins = np.percentile(ALL_KO_CORE, np.linspace(0, 100, 11))
decile_bins[0] = -np.inf
decile_bins[-1] = np.inf

def assign_decile(val):
    # returns decile index 0..9 based on bin edges
    return np.searchsorted(decile_bins[1:-1], val)

# ------------------------------------------------------------
# 3. Count primary KOs per decile
# ------------------------------------------------------------
primary_deciles = primary_coreness.apply(assign_decile)
decile_counts = primary_deciles.value_counts().to_dict()  # {decile: count}

# ------------------------------------------------------------
# 4. Build pool of non‑primary KOs per decile
# ------------------------------------------------------------
pool_by_decile = {}
for d in range(10):
    # Select all KOs in this decile, excluding primary KOs
    pool = ALL_KO_CORE[(ALL_KO_CORE.apply(assign_decile) == d) &
                       (~ALL_KO_CORE.index.isin(primary_kos))]
    pool_by_decile[d] = list(pool.index)

print('Decile pool sizes (excl. primary KOs):')
for d in range(10):
    needed = decile_counts.get(d, 0)
    print(f'  Decile {d}: {len(pool_by_decile[d]):,} KOs (need {needed} per permutation)')

# Optional: check that each decile has at least the needed number
for d in range(10):
    if len(pool_by_decile[d]) < decile_counts.get(d, 0):
        print(f'WARNING: Decile {d} has only {len(pool_by_decile[d])} KOs, but need {decile_counts.get(d,0)} – consider pooling adjacent deciles.')
        

Primary KOs with coreness data: 45 / 140
Decile pool sizes (excl. primary KOs):
  Decile 0: 671 KOs (need 4 per permutation)
  Decile 1: 663 KOs (need 3 per permutation)
  Decile 2: 662 KOs (need 2 per permutation)
  Decile 3: 665 KOs (need 3 per permutation)
  Decile 4: 662 KOs (need 5 per permutation)
  Decile 5: 664 KOs (need 5 per permutation)
  Decile 6: 664 KOs (need 4 per permutation)
  Decile 7: 664 KOs (need 4 per permutation)
  Decile 8: 659 KOs (need 8 per permutation)
  Decile 9: 661 KOs (need 7 per permutation)


In [13]:
def sample_matched_ko_set():
    """Sample a KO set matched to primary_kos on coreness decile distribution."""
    sampled = []
    for d, n in decile_counts.items():
        pool = pool_by_decile.get(d, [])
        if len(pool) < n:
            # if pool exhausted, sample with replacement
            sampled.extend(random.choices(pool, k=n) if pool else [])
        else:
            sampled.extend(random.sample(pool, n))
    return list(set(sampled))[:N_TARGET]  # ensure no duplicates, cap to N_TARGET

def run_density_pgls(ko_list, label):
    """Compute per-genus KO density via Spark + PGLS. Returns beta or None."""
    ko_prefixed = [f'ko:{k}' for k in ko_list]
    quoted = ', '.join(f"'{k}'" for k in ko_prefixed)
    sql = f"""
        SELECT gm.genome_id,
               regexp_extract(gm.lineage, 'g__([^;]+)', 1) AS genus,
               COUNT(DISTINCT koid.ko)                      AS n_ko,
               gm.length                                    AS genome_length_bp
        FROM kescience_mgnify.genome gm
        JOIN (
            SELECT genome_id, explode(split(kegg_ko, ',')) AS ko
            FROM kescience_mgnify.gene_eggnog
            WHERE kegg_ko IS NOT NULL AND kegg_ko != '-'
        ) koid USING (genome_id)
        WHERE koid.ko IN ({quoted})
        GROUP BY gm.genome_id, gm.lineage, gm.length
    """
    pm = _spark.sql(sql).toPandas()
    pm['genus_lower'] = pm['genus'].str.lower().str.strip()
    pm['ko_per_mb']   = pm['n_ko'] / (pm['genome_length_bp'] / 1e6)
    dens = pm.groupby('genus_lower', as_index=False).agg(ko_per_mb=('ko_per_mb','mean'))
    merged = trait_df.merge(dens, on='genus_lower', how='inner').copy()
    if len(merged) < 100:
        return None
    mu, sd = merged['ko_per_mb'].mean(), merged['ko_per_mb'].std()
    merged['ko_per_mb_z'] = (merged['ko_per_mb'] - mu) / sd
    try:
        res = run_pgls(merged.dropna(subset=['ko_per_mb_z','mean_levins_B_std']),
                       TREE_BAC, response='mean_levins_B_std',
                       predictors=['ko_per_mb_z'], taxon_col='genus_lower',
                       label=label, min_n=100)
        return res
    except Exception:
        return None


N_PERM = 1000
perm_results = []
print(f'Running {N_PERM} coreness-matched permutations (this will take time)...')
for i in range(N_PERM):
    kos = sample_matched_ko_set()
    res = run_density_pgls(kos, f'perm_{i:04d}')
    if res is not None:
        perm_results.append({'permutation': i, 'n_kos': len(kos),
                             'beta': res['beta'], 'SE': res['SE'],
                             'p_value': res['p_value'], 'n_genera': res['n'],
                             'lambda_est': res['lambda_est']})
    if (i+1) % 100 == 0:
        completed = [r['beta'] for r in perm_results if r is not None]
        print(f'  {i+1}/{N_PERM} done; valid={len(completed)}; '
              f'β range [{min(completed):.4f}, {max(completed):.4f}]')

perm_df = pd.DataFrame(perm_results)
emp_p = (perm_df['beta'] <= OBS_BETA).mean()
print(f'\nEmpirical p = {emp_p:.4g} ({(perm_df["beta"]<=OBS_BETA).sum()} / {len(perm_df)} permuted β ≤ {OBS_BETA})')
perm_df.to_csv(DATA / 'coreness_permutation_results.csv', index=False)
print('Saved: data/coreness_permutation_results.csv')


Running 1000 coreness-matched permutations (this will take time)...


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


...

/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  1000/1000 done; valid=1000; β range [-0.0385, 0.0118]

Empirical p = 0.298 (298 / 1000 permuted β ≤ -0.021)
Saved: data/coreness_permutation_results.csv


## Block 4 — Genome-size attenuation profile (100 sets)

In [15]:
def _extract_beta(res, focal='ko_per_mb_z'):
    """Extract β, SE, p for focal predictor; handles single- and multi-predictor run_pgls."""
    if 'beta' in res:
        return res['beta'], res['SE'], res['p_value']
    return res['betas'][focal], res['SEs'][focal], res['p_values'][focal]

# Genome size is already in bac_base (mean_genome_mb). No additional Spark query needed.
# gs_df is used in run_density_pgls_with_gs below — keep for that function's merge.
gs_df = bac_base[['genus_lower', 'mean_genome_mb']].dropna(subset=['mean_genome_mb']).copy()
print(f'Genome size available for {len(gs_df)} genera (from bac_base)')

# Primary-analysis attenuation: use bac_base directly (already has mean_genome_mb)
bac_with_gs_merged = bac_base.dropna(subset=['mean_genome_mb', 'mean_levins_B_std']).copy()
mu, sd = bac_with_gs_merged['ko_per_mb_primary'].mean(), bac_with_gs_merged['ko_per_mb_primary'].std()
bac_with_gs_merged['ko_per_mb_z'] = (bac_with_gs_merged['ko_per_mb_primary'] - mu) / sd
gs_mu, gs_sd = bac_with_gs_merged['mean_genome_mb'].mean(), bac_with_gs_merged['mean_genome_mb'].std()
bac_with_gs_merged['genome_size_z'] = (bac_with_gs_merged['mean_genome_mb'] - gs_mu) / gs_sd

res_no_gs = run_pgls(
    bac_with_gs_merged.dropna(subset=['ko_per_mb_z', 'mean_levins_B_std']),
    TREE_BAC, response='mean_levins_B_std',
    predictors=['ko_per_mb_z'], taxon_col='genus_lower',
    label='primary_no_gs', min_n=100)

res_with_gs = run_pgls(
    bac_with_gs_merged.dropna(subset=['ko_per_mb_z', 'genome_size_z', 'mean_levins_B_std']),
    TREE_BAC, response='mean_levins_B_std',
    predictors=['ko_per_mb_z', 'genome_size_z'], taxon_col='genus_lower',
    label='primary_with_gs', min_n=100)

beta_no_gs,   _, _ = _extract_beta(res_no_gs,   'ko_per_mb_z')
beta_with_gs, _, _ = _extract_beta(res_with_gs, 'ko_per_mb_z')
observed_attenuation = (beta_no_gs - beta_with_gs) / beta_no_gs * 100

print(f'Primary β without genome size: {beta_no_gs:+.4f}')
print(f'Primary β with genome size:    {beta_with_gs:+.4f}')
print(f'Observed attenuation:          {observed_attenuation:.1f}%')


Genome size available for 1574 genera (from bac_base)


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


Primary β without genome size: -0.0207
Primary β with genome size:    -0.0110
Observed attenuation:          46.7%


In [16]:
# Run 100 permutations with genome size covariate.
# gs_df_local provides per-genus mean_genome_mb (from bac_base — no Spark needed for genome size).
# Density per KO set still requires Spark.

def run_density_pgls_with_gs(ko_list, gs_df_local, label):
    ko_prefixed = [f'ko:{k}' for k in ko_list]
    quoted = ', '.join(f"'{k}'" for k in ko_prefixed)
    sql = f"""
        SELECT gm.genome_id,
               regexp_extract(gm.lineage, 'g__([^;]+)', 1) AS genus,
               COUNT(DISTINCT koid.ko)                      AS n_ko,
               gm.length                                    AS genome_length_bp
        FROM kescience_mgnify.genome gm
        JOIN (SELECT genome_id, explode(split(kegg_ko, ',')) AS ko
              FROM kescience_mgnify.gene_eggnog
              WHERE kegg_ko IS NOT NULL AND kegg_ko != '-') koid USING (genome_id)
        WHERE koid.ko IN ({quoted})
        GROUP BY gm.genome_id, gm.lineage, gm.length
    """
    pm = _spark.sql(sql).toPandas()
    pm['genus_lower'] = pm['genus'].str.lower().str.strip()
    pm['ko_per_mb']   = pm['n_ko'] / (pm['genome_length_bp'] / 1e6)
    dens = pm.groupby('genus_lower', as_index=False).agg(ko_per_mb=('ko_per_mb', 'mean'))
    merged = (trait_df
              .merge(dens, on='genus_lower', how='inner')
              .merge(gs_df_local[['genus_lower', 'mean_genome_mb']], on='genus_lower', how='inner')
              .copy())
    if len(merged) < 100:
        return None, None
    mu, sd = merged['ko_per_mb'].mean(), merged['ko_per_mb'].std()
    merged['ko_per_mb_z'] = (merged['ko_per_mb'] - mu) / sd
    gmu, gsd = merged['mean_genome_mb'].mean(), merged['mean_genome_mb'].std()
    merged['genome_size_z'] = (merged['mean_genome_mb'] - gmu) / gsd
    df_fit = merged.dropna(subset=['ko_per_mb_z', 'genome_size_z', 'mean_levins_B_std'])
    try:
        r1 = run_pgls(df_fit, TREE_BAC, response='mean_levins_B_std',
                      predictors=['ko_per_mb_z'], taxon_col='genus_lower',
                      label=f'{label}_no_gs', min_n=100)
        r2 = run_pgls(df_fit, TREE_BAC, response='mean_levins_B_std',
                      predictors=['ko_per_mb_z', 'genome_size_z'], taxon_col='genus_lower',
                      label=f'{label}_with_gs', min_n=100)
        return r1, r2
    except Exception:
        return None, None

N_ATT = 100
att_results = []
print(f'Running {N_ATT} permutations with genome-size covariate...')
perm_kos_sample = [sample_matched_ko_set() for _ in range(N_ATT)]
for i, kos in enumerate(perm_kos_sample):
    r1, r2 = run_density_pgls_with_gs(kos, gs_df, f'att_{i:03d}')
    if r1 is not None and r2 is not None:
        b1, _, _ = _extract_beta(r1, 'ko_per_mb_z')
        b2, _, _ = _extract_beta(r2, 'ko_per_mb_z')
        att_pct = (b1 - b2) / b1 * 100 if b1 != 0 else np.nan
        att_results.append({'permutation': i, 'beta_no_gs': b1,
                            'beta_with_gs': b2, 'attenuation_pct': att_pct})
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{N_ATT}')

att_df = pd.DataFrame(att_results)
att_df['observed_attenuation_pct'] = observed_attenuation
att_df.to_csv(DATA / 'attenuation_profile_comparison.csv', index=False)
print(f'\nSaved: data/attenuation_profile_comparison.csv ({len(att_df)} rows)')
print(f'Permuted attenuation median: {att_df["attenuation_pct"].median():.1f}%')
print(f'Observed attenuation:        {observed_attenuation:.1f}%')


Running 100 permutations with genome-size covariate...


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


...

/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  100/100

Saved: data/attenuation_profile_comparison.csv (100 rows)
Permuted attenuation median: 94.9%
Observed attenuation:        46.7%


## Block 5 — Figures and REPORT paragraph

In [17]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

BLUE = '#0072B2'; GREY = '#999999'; ORANGE = '#E69F00'

perm_df = pd.read_csv(DATA / 'coreness_permutation_results.csv')
att_df  = pd.read_csv(DATA / 'attenuation_profile_comparison.csv')

emp_p = (perm_df['beta'] <= OBS_BETA).mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Panel A: permutation null distribution
ax1.hist(perm_df['beta'], bins=50, color=GREY, alpha=0.75, edgecolor='white', lw=0.4)
ax1.axvline(OBS_BETA, color=BLUE, lw=2.5,
            label=f'Observed β = {OBS_BETA:.3f}\nemp p = {emp_p:.4f}')
ax1.set_xlabel('PGLS β (permuted KO set)', fontsize=10)
ax1.set_ylabel('Count', fontsize=10)
ax1.set_title('A. Coreness-matched permutation null\n(n=1,000 matched KO sets)', fontsize=10)
ax1.legend(fontsize=9, framealpha=0.9)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

# Panel B: attenuation profile comparison
ax2.hist(att_df['attenuation_pct'].dropna(), bins=30, color=GREY, alpha=0.75,
         edgecolor='white', lw=0.4, label='Permuted sets (n=100)')
ax2.axvline(observed_attenuation, color=BLUE, lw=2.5,
            label=f'Observed = {observed_attenuation:.1f}%')
ax2.set_xlabel('% β-attenuation by genome size covariate', fontsize=10)
ax2.set_ylabel('Count', fontsize=10)
ax2.set_title('B. Genome-size attenuation profile\n(permuted vs. observed metal gene set)', fontsize=10)
ax2.legend(fontsize=9, framealpha=0.9)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(str(FIGS / 'coreness_permutation_histogram.png'), dpi=300, bbox_inches='tight')
plt.close()
print('Saved: figures/coreness_permutation_histogram.png')

print('\n=== REPORT.md paragraph for NB20 ===')
print(f"""
To confirm that KO coreness (the fraction of genomes in the pan-genome carrying a
given KO) cannot explain the primary signal, we generated 1,000 permuted KO sets
matched to the primary set (n={N_TARGET}) on both KO count and coreness distribution
(decile matching within the full kbase.ke_pangenome KO repertoire). Running the
identical PGLS pipeline on each permuted set yielded a null distribution of β values
centred near zero (median = [X]), with the observed primary β = {OBS_BETA} falling
below {100*(1-emp_p):.1f}% of permuted values (empirical p = {emp_p:.4g}).

As an additional check, we ran a parallel set of 100 coreness-matched permutations
with genome size as a covariate and computed the % β-attenuation for each. The
observed attenuation of {observed_attenuation:.1f}% (β without genome size =
[X], with = [Y]) fell [within / outside] the central 95% of the permuted
attenuation distribution (permuted median = [Z]%), indicating that genome-size
confounding affects the primary set to a [similar / different] degree as matched
random KO sets.
""")


Saved: figures/coreness_permutation_histogram.png

=== REPORT.md paragraph for NB20 ===

To confirm that KO coreness (the fraction of genomes in the pan-genome carrying a
given KO) cannot explain the primary signal, we generated 1,000 permuted KO sets
matched to the primary set (n=140) on both KO count and coreness distribution
(decile matching within the full kbase.ke_pangenome KO repertoire). Running the
identical PGLS pipeline on each permuted set yielded a null distribution of β values
centred near zero (median = [X]), with the observed primary β = -0.021 falling
below 70.2% of permuted values (empirical p = 0.298).

As an additional check, we ran a parallel set of 100 coreness-matched permutations
with genome size as a covariate and computed the % β-attenuation for each. The
observed attenuation of 46.7% (β without genome size =
[X], with = [Y]) fell [within / outside] the central 95% of the permuted
attenuation distribution (permuted median = [Z]%), indicating that genome-size
co